# PyTorch 环境检测（修复版）
**必须从 Cell 1 开始按顺序运行，不能跳过。**

In [1]:
# Cell 1：必须第一个运行 —— 设置环境变量，防止 kernel crash
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'  # MPS 不支持的 op 自动回退 CPU
print('✅ 环境变量设置完成')

✅ 环境变量设置完成


In [2]:
# Cell 2：基础信息
import torch
import platform

print(f'Python:        {platform.python_version()}')
print(f'PyTorch:       {torch.__version__}')
print(f'CUDA 可用:     {torch.cuda.is_available()}')
print(f'MPS 可用:      {torch.backends.mps.is_available()}')

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'\n✅ 使用设备:   {device}')

Python:        3.11.15
PyTorch:       2.11.0
CUDA 可用:     False
MPS 可用:      True

✅ 使用设备:   mps


In [3]:
# Cell 3：Tensor 操作 + 设备转移
a = torch.randn(3, 3)
b = torch.randn(3, 3)

a_dev = a.to(device)
b_dev = b.to(device)
c_dev = torch.matmul(a_dev, b_dev)

print(f'Tensor a (CPU):\n{a}')
print(f'\na @ b 结果 (在 {device} 上计算):\n{c_dev}')
print(f'\n结果 device: {c_dev.device}')
print('\n✅ Tensor 操作 + 设备转移正常')

Tensor a (CPU):
tensor([[ 0.8151, -1.2563, -0.5049],
        [-1.4980,  0.1696,  0.8369],
        [ 0.1651, -0.1207, -0.0087]])

a @ b 结果 (在 mps 上计算):
tensor([[-0.3251,  3.1488,  1.8262],
        [ 0.5169, -4.2100, -1.3882],
        [-0.0793,  0.3336,  0.2612]], device='mps:0')

结果 device: mps:0

✅ Tensor 操作 + 设备转移正常


In [4]:
# Cell 4：Autograd 自动求导
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1   # y = (x+1)², dy/dx = 2x+2
y.backward()

print(f'y = x² + 2x + 1,  x = {x.item():.1f}')
print(f'dy/dx 解析值:    {2*x.item() + 2:.1f}')
print(f'dy/dx autograd: {x.grad.item():.1f}')
assert abs(x.grad.item() - 8.0) < 1e-5
print('\n✅ Autograd 正常')

y = x² + 2x + 1,  x = 3.0
dy/dx 解析值:    8.0
dy/dx autograd: 8.0

✅ Autograd 正常


In [5]:
# Cell 5：完整训练循环（拟合 y = 2x + 1）
import torch.nn as nn
torch.manual_seed(42)

X = torch.linspace(-1, 1, 100).unsqueeze(1).to(device)
Y = 2 * X + 1 + 0.05 * torch.randn_like(X)

model = nn.Linear(1, 1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

for epoch in range(500):
    pred = model(X)
    loss = loss_fn(pred, Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

w = model.weight.item()
b = model.bias.item()
print(f'拟合结果: y = {w:.4f}x + {b:.4f}')
print(f'真实参数: y = 2.0000x + 1.0000')
print(f'最终 Loss: {loss.item():.6f}')
assert abs(w - 2.0) < 0.05 and abs(b - 1.0) < 0.05
print(f'\n✅ 神经网络训练正常（设备: {device}）')

拟合结果: y = 2.0031x + 1.0079
真实参数: y = 2.0000x + 1.0000
最终 Loss: 0.002217

✅ 神经网络训练正常（设备: mps）


In [6]:
# Cell 6：CPU vs MPS/GPU 速度对比
import time

size = 2000
a = torch.randn(size, size)
b = torch.randn(size, size)

# CPU
t0 = time.time()
for _ in range(10):
    torch.matmul(a, b)
cpu_time = (time.time() - t0) / 10 * 1000

# GPU/MPS
a_dev = a.to(device)
b_dev = b.to(device)
torch.matmul(a_dev, b_dev)  # warm-up

t0 = time.time()
for _ in range(10):
    torch.matmul(a_dev, b_dev)
gpu_time = (time.time() - t0) / 10 * 1000

print(f'矩阵乘法 {size}x{size}，平均 10 次')
print(f'CPU  耗时: {cpu_time:.2f} ms')
print(f'{str(device).upper()} 耗时: {gpu_time:.2f} ms')
if device.type != "cpu":
    print(f'加速比: {cpu_time/gpu_time:.1f}x')
print('\n✅ 速度测试完成')

矩阵乘法 2000x2000，平均 10 次
CPU  耗时: 11.03 ms
MPS 耗时: 0.03 ms
加速比: 350.2x

✅ 速度测试完成


In [7]:
# Cell 7：汇总
print('========== 环境检测汇总 ==========')
print(f'  PyTorch 版本: {torch.__version__}')
print(f'  计算设备:     {device}')
print(f'  Tensor 操作:  ✅')
print(f'  Autograd:     ✅')
print(f'  模型训练:     ✅')
print(f'  GPU 加速:     {"✅" if device.type != "cpu" else "⚠️  未检测到 GPU，使用 CPU"}')
print('\n🎉 环境一切正常，可以开始学习 PyTorch！')

========== 环境检测汇总 ==========
  PyTorch 版本: 2.11.0
  计算设备:     mps
  Tensor 操作:  ✅
  Autograd:     ✅
  模型训练:     ✅
  GPU 加速:     ✅

🎉 环境一切正常，可以开始学习 PyTorch！


In [8]:
import torchvision
import torchvision.transforms as transforms
from torchvision import models

print(f'torchvision 版本: {torchvision.__version__}')

# 1. 测试 transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
print('✅ transforms 正常')

# 2. 加载预训练模型结构（不下载权重）
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

resnet = models.resnet18(weights=None).to(device)
resnet.eval()
print('✅ ResNet18 模型结构加载正常')

# 3. 随机 tensor 模拟图片推理
dummy_image = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    output = resnet(dummy_image)
print(f'✅ 推理正常，输出形状: {output.shape}')  # 应为 [1, 1000]

print(f'\n✅ torchvision 全部验证通过（设备: {device}）')

torchvision 版本: 0.26.0
✅ transforms 正常
✅ ResNet18 模型结构加载正常
✅ 推理正常，输出形状: torch.Size([1, 1000])

✅ torchvision 全部验证通过（设备: mps）
